# 02 — Coleta e preparação do nível do Guaíba

**Fonte:** ANA, serviço público [DadosHidrometeorologicos](https://telemetriaws1.ana.gov.br/ServiceANA.asmx?WSDL), estação **87450020 — Usina do Gasômetro**, série corrigida para o Marco IBGE 1788A. A [ANA/SGB documenta a série](https://www.gov.br/ana/pt-br/assuntos/noticias-e-eventos/noticias/ana-e-servico-geologico-do-brasil-fazem-readequacoes-em-estacao-de-monitoramento-do-rio-guaiba-em-porto-alegre-rs).

Recorte: 03/05 a 30/06/2024, Porto Alegre; medições de nível em centímetros convertidas para metros. O script obtém o XML por código e período, guarda o bruto em cache e registra a data de coleta. Esta é uma análise **independente** da simulação do INCT.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
FIG = ROOT / 'resultados' / 'figuras'
FIG.mkdir(parents=True, exist_ok=True)

import pandas as pd
from scripts.preparar_dados import clean_ana
ana, audit = clean_ana()
print(f'{len(ana)} horários na grade; {ana.nivel_m.notna().sum()} com nível; {len(audit)} lacunas auditadas')
display(ana.head())

5664 horários na grade; 5444 com nível; 220 lacunas auditadas


/Users/lambari.exe/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,data_hora,estacao_codigo,estacao_nome,nivel_cm,chuva_mm,nivel_m,referencial
0,2024-05-03 00:00:00,87450020,"Usina do Gasômetro, referencial corrigido",NaN,0.0,NaN,Marco IBGE 1788A; nível médio do mar
1,2024-05-03 00:15:00,87450020,"Usina do Gasômetro, referencial corrigido",NaN,0.0,NaN,Marco IBGE 1788A; nível médio do mar
2,2024-05-03 00:30:00,87450020,"Usina do Gasômetro, referencial corrigido",NaN,0.0,NaN,Marco IBGE 1788A; nível médio do mar
3,2024-05-03 00:45:00,87450020,"Usina do Gasômetro, referencial corrigido",NaN,0.0,NaN,Marco IBGE 1788A; nível médio do mar
4,2024-05-03 01:00:00,87450020,"Usina do Gasômetro, referencial corrigido",NaN,0.0,NaN,Marco IBGE 1788A; nível médio do mar


## Qualidade e tratamento

Datas inválidas e duplicatas são removidas se existirem. Níveis ausentes ficam vazios: interpolá-los poderia criar valores artificiais próximos ao pico. Mudanças abruptas são listadas para inspeção, mas não são apagadas automaticamente, pois podem refletir variação real ou ruído de medição.

In [2]:
valid = ana.dropna(subset=['nivel_cm']).sort_values('data_hora').copy()
valid['variacao_cm'] = valid.nivel_cm.diff()
print('Registros sem nível:', audit.problema.eq('nível ausente').sum())
print('Horários sem registro:', audit.problema.eq('horário sem registro no XML').sum())
print('Máxima variação absoluta consecutiva (cm):', valid.variacao_cm.abs().max())
display(valid.loc[valid.variacao_cm.abs().nlargest(10).index, ['data_hora','nivel_cm','variacao_cm']])
display(pd.read_csv(ROOT / 'dados/originais/ana_metadados_coleta.csv'))

Registros sem nível: 107
Horários sem registro: 113
Máxima variação absoluta consecutiva (cm): 8.0


,data_hora,nivel_cm,variacao_cm
96,2024-05-04 00:00:00,370.0,8.0
215,2024-05-05 05:45:00,409.0,-8.0
243,2024-05-05 12:45:00,407.0,-8.0
183,2024-05-04 21:45:00,409.0,7.0
214,2024-05-05 05:30:00,417.0,7.0
224,2024-05-05 08:00:00,415.0,7.0
622,2024-05-09 11:30:00,377.0,-7.0
108,2024-05-04 03:00:00,376.0,6.0
120,2024-05-04 06:00:00,380.0,6.0
156,2024-05-04 15:00:00,398.0,6.0


,estacao_codigo,estacao_nome,inicio_solicitado,fim_solicitado,linhas_xml,arquivo_bruto,coletado_ou_cache_em,url,licenca
0,87450020,"Usina do Gasômetro, referencial corrigido",2024-05-03,2024-06-30,5551,ana_87450020_2024-05-03_2024-06-30.xml,2026-10-26T20:35:49,https://telemetriaws1.ana.gov.br/ServiceANA.as...,Consultar termos da ANA; fonte pública oficial...


In [3]:
peak = valid.loc[valid.nivel_m.idxmax()]
print(f'Pico na série: {peak.nivel_m:.2f} m em {peak.data_hora}')
assert ana.estacao_codigo.astype(str).eq('87450020').all()
assert ana.data_hora.is_unique
print('Arquivos salvos em dados/originais/ e dados/processados/.')

Pico na série: 4.17 m em 2024-05-05 05:30:00
Arquivos salvos em dados/originais/ e dados/processados/.
